# Cascade JEPA on a frozen decoder (Qwen3-1.7B), NOUL questions

Nothing in the decoder is trained. It reads each input once; only small heads train on its hidden states, so the JEPA comparison is not mixed up with fine-tuning.

| Arm | Trained | Shows |
|---|---|---|
| **Z** zero-shot | nothing: the model's own P(yes) / P(no) after the prompt | what the decoder knows untrained |
| **D** plain head | predictor + 2 logits on the frozen states | the no-JEPA baseline |
| **A** cascade | the same predictor, stage 1 to Q·S first, then the decision (cosine to the yes/no vectors) | **A − D: JEPA** |
| **C** shuffled Q·S | as A, with another question's Q·S as the stage-1 target | **A − C: the Q·S content** |

- **Online input:** the chat prompt (state, question, "Answer yes or no.") through the frozen decoder; the predictor reads layer `layer`'s token states.
- **Q·S:** the question alone (q: mean of its token states) attending over the state alone (its token states), at the same layer; the recipe is chosen by the check (standardised space, sharpened attention, minus the state's plain average), which must show that Q·S differs between questions on the same state.
- **Choice vectors:** `yes` × 64 and `no` × 64 at the same layer.
- **Matched training:** D gets as many epochs as A's stage 1 + stage 2 together. The best decision epoch by validation KL is kept.

Measures: validation, our test split, and the 600 Typed Decisions NOUL questions (Signpost-D-4B: 0.783 accuracy).

## Settings

In [ ]:
import os


def setting(name, default):
    value = os.environ.get(name.upper())
    if value is None:
        return default
    if isinstance(default, list):
        kind = type(default[0]) if default else str
        return [kind(v.strip()) for v in value.split(",") if v.strip()]
    if isinstance(default, bool):
        return value.strip().lower() in ("1", "true", "yes")
    return type(default)(value)


CONFIG = {
    "seed": setting("seed", 42),
    "model_name": setting("model_name", "Qwen/Qwen3-1.7B"),
    "layer": setting("layer", 20),                      # of 28: a middle-to-late layer
    "data_path": setting("data_path", "data/signpost_d_decisions/train.jsonl"),
    "data_url": setting("data_url", "https://raw.githubusercontent.com/skamalj/llms-the-hard-way/"
                                    "decider-jepa/data/signpost_d_decisions/train.jsonl"),
    "max_sequence_length": setting("max_sequence_length", 1536),
    "split_seed": setting("split_seed", 42),
    "train_ratio": setting("train_ratio", 0.80),
    "validation_ratio": setting("validation_ratio", 0.10),
    "encode_batch": setting("encode_batch", 8),

    "qs_sharpness": setting("qs_sharpness", [1.0, 4.0, 16.0]),
    "qs_gate_same_state_cos": setting("qs_gate_same_state_cos", 0.95),

    "target_words": setting("target_words", ["yes", "no"]),
    "target_repeats": setting("target_repeats", 64),
    "temperature": setting("temperature", 0.1),

    "pred_width": setting("pred_width", 384),
    "pred_layers": setting("pred_layers", 2),
    "pred_heads": setting("pred_heads", 6),
    "dropout": setting("dropout", 0.1),

    "batch_size": setting("batch_size", 32),
    "lr": setting("lr", 3e-4),
    "weight_decay": setting("weight_decay", 0.01),
    "stage1_epochs": setting("stage1_epochs", 10),
    "stage2_epochs": setting("stage2_epochs", 20),
    "stage1_weight_in_stage2": setting("stage1_weight_in_stage2", 1.0),
    "arms": setting("arms", ["D", "A", "C"]),

    "td_dataset": setting("td_dataset", "LocalLLaMA/typed-decisions"),
    "td_file": setting("td_file", "all/test-00000-of-00001.parquet"),
    "out_dir": setting("out_dir", "cascade_decoder"),
}

for k, v in CONFIG.items():
    print(f"{k:26s}: {v}")

## Data (as in the other NOUL notebooks)

In [ ]:
import json
import random
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from huggingface_hub import hf_hub_download

random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
device = torch.device("cuda")

DATA_PATH = Path(CONFIG["data_path"])
if not DATA_PATH.exists():
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(CONFIG["data_url"], DATA_PATH)


def parsed(value):
    return json.loads(value) if isinstance(value, str) else value


def noul_samples(records):
    out = []
    for record in records:
        state_text = "State:\n" + str(record["state"])
        for qid, q in record["questions"].items():
            if str(q.get("type", "")).lower() != "noul":
                continue
            gold = {str(k): float(v) for k, v in record["gold"][qid]["probabilities"].items()}
            question_text = "Question:\n" + str(q.get("instructions"))
            out.append({"group": record["id"], "question_id": qid,
                        "state_text": state_text, "question_text": question_text,
                        "gold": [gold["true"], gold["false"]]})
    return out


records = [json.loads(line) for line in DATA_PATH.open(encoding="utf-8") if line.strip()]
samples = noul_samples(records)

groups = pd.Series([s["group"] for s in samples]).drop_duplicates().to_numpy()
shuffled = np.random.default_rng(CONFIG["split_seed"]).permutation(groups)
n_train = int(len(groups) * CONFIG["train_ratio"])
n_val = int(len(groups) * CONFIG["validation_ratio"])
split_of = {g: "train" for g in shuffled[:n_train]}
split_of.update({g: "validation" for g in shuffled[n_train:n_train + n_val]})
split_of.update({g: "test" for g in shuffled[n_train + n_val:]})
splits = {name: [s for s in samples if split_of[s["group"]] == name] for name in ("train", "validation", "test")}

td_frame = pd.read_parquet(hf_hub_download(CONFIG["td_dataset"], CONFIG["td_file"], repo_type="dataset"))
td_records = [{"id": r["id"], "state": parsed(r["state"]), "questions": parsed(r["questions"]),
               "gold": parsed(r["gold"])} for r in td_frame.to_dict("records")]
splits["td_test"] = noul_samples(td_records)

GOLD = {name: torch.tensor([r["gold"] for r in rows]) for name, rows in splits.items()}
print({k: len(v) for k, v in splits.items()})

## The frozen decoder: one pass per input

- **Prompt:** Qwen3's chat template, no thinking: the state, the question, then "Answer yes or no."
- **Zero-shot (arm Z):** from the same pass, P(yes) = softmax over the next-token logits of {`Yes`, `yes`} against {`No`, `no`}.
- **Kept:** layer `layer`'s token states of the prompt (bf16, on the GPU), the question alone (mean), the state alone (its tokens), and the yes/no × 64 vectors.

In [ ]:
import math
import time

import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])
decoder = AutoModelForCausalLM.from_pretrained(CONFIG["model_name"], dtype=torch.bfloat16).to(device).eval()
HIDDEN = decoder.config.hidden_size
LAYER = CONFIG["layer"]
print(f"{CONFIG['model_name']}: {decoder.config.num_hidden_layers} layers, hidden {HIDDEN}, reading layer {LAYER}")


def prompt_of(row):
    messages = [{"role": "user", "content": row["state_text"] + "\n\n" + row["question_text"] + "\n\nAnswer yes or no."}]
    return tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False, enable_thinking=False)


def single_token(text):
    ids = tokenizer.encode(text, add_special_tokens=False)
    assert len(ids) == 1, (text, ids)
    return ids[0]


YES_IDS = [single_token(t) for t in ("Yes", "yes")]
NO_IDS = [single_token(t) for t in ("No", "no")]

# show one prompt
print(prompt_of(splits["train"][0])[-300:])


@torch.no_grad()
def run(texts, keep_all_tokens):
    """-> (layer states per text [fp16, GPU], zero-shot log-odds per text or None)."""
    states_out, zero_shot = [None] * len(texts), [None] * len(texts)
    order = sorted(range(len(texts)), key=lambda i: len(texts[i]))
    for s in range(0, len(order), CONFIG["encode_batch"]):
        idx = order[s:s + CONFIG["encode_batch"]]
        batch = tokenizer([texts[i] for i in idx], padding=True, truncation=True, add_special_tokens=False,
                          max_length=CONFIG["max_sequence_length"], return_tensors="pt").to(device)
        out = decoder(**batch, output_hidden_states=True)
        layer_states = out.hidden_states[LAYER]
        lengths = batch["attention_mask"].sum(1)
        for row, i in enumerate(idx):
            n = int(lengths[row])
            states_out[i] = layer_states[row, :n].clone() if keep_all_tokens else layer_states[row, :n].float().mean(0)
            last = out.logits[row, n - 1].float()
            zero_shot[i] = (torch.logsumexp(last[YES_IDS], 0) - torch.logsumexp(last[NO_IDS], 0)).item()
        del out
    return states_out, zero_shot


tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

t0 = time.perf_counter()
JOINT, ZERO_SHOT, Q, S = {}, {}, {}, {}
for name, rows in splits.items():
    JOINT[name], ZERO_SHOT[name] = run([prompt_of(r) for r in rows], keep_all_tokens=True)
    q, _ = run([r["question_text"] for r in rows], keep_all_tokens=False)
    Q[name] = torch.stack(q)
    unique_states = sorted({r["state_text"] for r in rows})
    state_tokens, _ = run(unique_states, keep_all_tokens=True)
    by_text = dict(zip(unique_states, state_tokens))
    S[name] = [by_text[r["state_text"]] for r in rows]
    print(f"{name:10s}: {len(rows)} questions, {len(unique_states)} states, prompt tokens median "
          f"{int(np.median([len(t) for t in JOINT[name]]))} | {time.perf_counter() - t0:.0f} s")

choice_vectors, _ = run([" ".join([w] * CONFIG["target_repeats"]) for w in CONFIG["target_words"]],
                        keep_all_tokens=False)
CHOICES = torch.stack(choice_vectors).float()
c = F.cosine_similarity(CHOICES[0], CHOICES[1], dim=0).item()
print(f"choice vectors: cos {c:.4f}, highest gold probability {1 / (1 + math.exp(-(1 - c) / CONFIG['temperature'])):.4f}")

del decoder
torch.cuda.empty_cache()
print(f"GPU memory in use: {torch.cuda.memory_allocated() / 2**30:.1f} GB")

## Arm Z: zero-shot

In [ ]:
def kl(logits, gold):
    return (gold * (gold.clamp_min(1e-12).log() - F.log_softmax(logits.float(), dim=1))).sum(1)


def measures(logits, gold):
    p = F.softmax(logits.float(), dim=1)
    return {"kl": kl(logits, gold).mean().item(), "uniform_kl": kl(torch.zeros_like(gold), gold).mean().item(),
            "top_choice": (p.argmax(1) == gold.argmax(1)).float().mean().item(),
            "brier": ((p - gold) ** 2).sum(1).mean().item(), "p_true_std": p[:, 0].std().item()}


results = {"Z": {}}
for name in splits:
    log_odds = torch.tensor(ZERO_SHOT[name])
    logits = torch.stack([log_odds / 2, -log_odds / 2], dim=1)       # P(yes) = sigmoid(log-odds)
    results["Z"][name] = measures(logits, GOLD[name])

display(pd.DataFrame(results["Z"]).T.round(4))

## Q·S and the check

The variants, as in `noul_cascade_jepa.ipynb`: raw or standardised space × attention sharpness × with or without subtracting the state's plain average. The variant with the lowest same-state cosine is used, and it must pass the gate.

In [ ]:
def running_stats(tensors):
    """Per-dimension mean and std over all rows of many [tokens, hidden] tensors, without concatenating them."""
    total = torch.zeros(HIDDEN, dtype=torch.float64, device=device)
    squares = torch.zeros(HIDDEN, dtype=torch.float64, device=device)
    count = 0
    for t in tensors:
        x = t.double()
        total += x.sum(0)
        squares += (x * x).sum(0)
        count += len(x)
    mean = total / count
    std = (squares / count - mean * mean).clamp_min(1e-12).sqrt()
    return mean.float(), std.float().clamp_min(1e-6)


TOKEN_MEAN, TOKEN_STD = running_stats({id(t): t for t in S["train"]}.values())


def qs_target(q, Sx, variant):
    space, sharpness, residual = variant
    Sx, q = Sx.float(), q.float()
    if space == "standardised":
        Sx = (Sx - TOKEN_MEAN) / TOKEN_STD
        q = (q - TOKEN_MEAN) / TOKEN_STD
    w = torch.softmax(sharpness * (Sx @ q) / math.sqrt(Sx.shape[1]), dim=0)
    entropy = -(w * w.clamp_min(1e-12).log()).sum() / math.log(max(len(w), 2))
    t = w @ Sx
    return (t - Sx.mean(0) if residual else t), entropy.item()


def qs_all(name, variant):
    out = [qs_target(Q[name][i], S[name][i], variant) for i in range(len(S[name]))]
    return torch.stack([t for t, _ in out]), np.array([h for _, h in out])


by_group = {}
for i, r in enumerate(splits["train"]):
    by_group.setdefault(r["group"], []).append(i)
pairs = np.random.default_rng(0).integers(0, len(splits["train"]), size=(3000, 2))


def qs_check(variant):
    t, h = qs_all("train", variant)
    same = [F.cosine_similarity(t[a], t[b], dim=0).item()
            for idx in by_group.values() for j, a in enumerate(idx) for b in idx[j + 1:]]
    other = [F.cosine_similarity(t[a], t[b], dim=0).item() for a, b in pairs
             if splits["train"][a]["group"] != splits["train"][b]["group"]]
    return {"space": variant[0], "sharpness": variant[1], "residual": variant[2],
            "same_state_cos": np.mean(same), "different_state_cos": np.mean(other), "attention_entropy": h.mean()}


variants = [(sp, sh, res) for sp in ("raw", "standardised") for sh in CONFIG["qs_sharpness"] for res in (False, True)]
qs_report = pd.DataFrame([qs_check(v) for v in variants]).sort_values("same_state_cos").reset_index(drop=True)
display(qs_report.round(4))

best = qs_report.iloc[0]
QS_VARIANT = (best["space"], float(best["sharpness"]), bool(best["residual"]))
print(f"Q·S variant {QS_VARIANT}: same state {best['same_state_cos']:.4f} | different states "
      f"{best['different_state_cos']:.4f} (gate < {CONFIG['qs_gate_same_state_cos']})")
assert best["same_state_cos"] < CONFIG["qs_gate_same_state_cos"], "Q·S does not carry the question."
print("Q·S PASSES THE CHECK")

QS = {name: qs_all(name, QS_VARIANT)[0].to(device) for name in splits}
QS_SHUFFLED = QS["train"][torch.from_numpy(np.random.default_rng(CONFIG["seed"]).permutation(len(QS["train"])))]
CHOICES = CHOICES.to(device)

mean_qs = QS["train"].mean(0)
for name in ("validation", "td_test"):
    print(f"{name}: cos(Q·S, mean training Q·S) = "
          f"{F.cosine_similarity(QS[name], mean_qs.unsqueeze(0), dim=1).mean().item():.4f}")

## The heads

The predictor reads the prompt's layer states (a learned query token and `pred_layers` transformer layers) → ẑ₁. Decision head: `cosine` (A, C), ẑ₁ → MLP → ẑ₂ scored against the yes/no vectors; or `logits` (D), ẑ₁ → MLP → 2 logits. Inputs are standardised per dimension with the training prompts' statistics (decoder states have a few very large dimensions).

In [ ]:
import copy

import torch.nn as nn

IN_MEAN, IN_STD = running_stats(JOINT["train"])


class Heads(nn.Module):
    def __init__(self, head):
        super().__init__()
        w = CONFIG["pred_width"]
        self.head_kind = head
        self.inp = nn.Linear(HIDDEN, w)
        self.query = nn.Parameter(torch.randn(1, 1, w) * 0.02)
        layer = nn.TransformerEncoderLayer(w, CONFIG["pred_heads"], 4 * w, CONFIG["dropout"],
                                           batch_first=True, norm_first=True, activation="gelu")
        self.layers = nn.TransformerEncoder(layer, CONFIG["pred_layers"])
        self.z1 = nn.Sequential(nn.LayerNorm(w), nn.Linear(w, HIDDEN))
        out = HIDDEN if head == "cosine" else 2
        self.head = nn.Sequential(nn.LayerNorm(HIDDEN), nn.Linear(HIDDEN, HIDDEN), nn.GELU(), nn.Linear(HIDDEN, out))

    def forward(self, tokens, pad):
        x = self.inp((tokens - IN_MEAN) / IN_STD)
        x = torch.cat([self.query.expand(len(x), -1, -1), x], dim=1)
        pad = torch.cat([torch.zeros(len(x), 1, dtype=torch.bool, device=x.device), pad], dim=1)
        z1 = self.z1(self.layers(x, src_key_padding_mask=pad)[:, 0])
        h = self.head(z1)
        if self.head_kind == "cosine":
            return z1, F.cosine_similarity(h.unsqueeze(1), CHOICES.unsqueeze(0), dim=-1) / CONFIG["temperature"]
        return z1, h


def batch_of(name, idx):
    seqs = [JOINT[name][i] for i in idx]
    n = max(len(t) for t in seqs)
    tokens = torch.zeros(len(seqs), n, HIDDEN, device=device)
    pad = torch.ones(len(seqs), n, dtype=torch.bool, device=device)
    for r, t in enumerate(seqs):
        tokens[r, :len(t)] = t.float()
        pad[r, :len(t)] = False
    return tokens, pad


@torch.no_grad()
def evaluate(model, name):
    model.eval()
    n = len(JOINT[name])
    order = sorted(range(n), key=lambda i: len(JOINT[name][i]))
    cos_all, logits_all = [], []
    for s in range(0, n, CONFIG["batch_size"]):
        idx = order[s:s + CONFIG["batch_size"]]
        z1, logits = model(*batch_of(name, idx))
        cos_all.append(F.cosine_similarity(z1, QS[name][idx], dim=1))
        logits_all.append(logits)
    logits = torch.cat(logits_all)
    gold = GOLD[name][order].to(device)
    return {"stage1_cos": torch.cat(cos_all).mean().item(), **measures(logits, gold)}

## Training the arms (matched epochs)

In [ ]:
ARM_SPEC = {"D": ("logits", None), "A": ("cosine", "real"), "C": ("cosine", "shuffled")}
total_epochs = CONFIG["stage1_epochs"] + CONFIG["stage2_epochs"]
history = []

for arm in CONFIG["arms"]:
    head, stage1 = ARM_SPEC[arm]
    torch.manual_seed(CONFIG["seed"])
    model = Heads(head).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
    target = QS["train"] if stage1 == "real" else QS_SHUFFLED
    best = (float("inf"), None, 0)
    n = len(JOINT["train"])
    for epoch in range(1, total_epochs + 1):
        in_stage1 = stage1 is not None and epoch <= CONFIG["stage1_epochs"]
        w1 = 0.0 if stage1 is None else (1.0 if in_stage1 else CONFIG["stage1_weight_in_stage2"])
        w_kl = 0.0 if in_stage1 else 1.0
        model.train()
        perm = torch.randperm(n, generator=torch.Generator().manual_seed(CONFIG["seed"] + epoch)).tolist()
        total = 0.0
        for s in range(0, n, CONFIG["batch_size"]):
            idx = perm[s:s + CONFIG["batch_size"]]
            z1, logits = model(*batch_of("train", idx))
            loss = w_kl * kl(logits, GOLD["train"][idx].to(device)).mean()
            if w1:
                loss = loss + w1 * (1 - F.cosine_similarity(z1, target[idx], dim=1)).mean()
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total += loss.item() * len(idx)
        v = evaluate(model, "validation")
        history.append({"arm": arm, "epoch": epoch, "stage1": in_stage1, "train_loss": total / n, **v})
        if epoch == 1 or epoch % 5 == 0 or epoch == CONFIG["stage1_epochs"] + 1:
            print(f"arm {arm} epoch {epoch:2d} {'(stage 1)' if in_stage1 else '(decision)'} | train {total / n:.4f} | "
                  f"val stage-1 cos {v['stage1_cos']:.4f} | val KL {v['kl']:.4f} (uniform {v['uniform_kl']:.4f}) | "
                  f"top {v['top_choice']:.3f} | P(true) std {v['p_true_std']:.3f}")
        if not in_stage1 and v["kl"] < best[0]:
            best = (v["kl"], copy.deepcopy(model.state_dict()), epoch)
    model.load_state_dict(best[1])
    results[arm] = {name: evaluate(model, name) for name in splits}
    results[arm]["best_epoch"] = best[2]
    r = results[arm]
    print(f"ARM {arm} RESULT (best epoch {best[2]}) | validation top {r['validation']['top_choice']:.4f} KL "
          f"{r['validation']['kl']:.4f} | test top {r['test']['top_choice']:.4f} KL {r['test']['kl']:.4f} | "
          f"Typed Decisions NOUL accuracy {r['td_test']['top_choice']:.4f} KL {r['td_test']['kl']:.4f}\n")

## Results

In [ ]:
rows = []
for arm, r in results.items():
    for name in ("train", "validation", "test", "td_test"):
        rows.append({"arm": arm, "split": name, "best_epoch": r.get("best_epoch", ""), **r[name]})
table = pd.DataFrame(rows)
pd.set_option("display.width", 250)
display(table.round(4))

print("Typed Decisions NOUL (600): Signpost-D-4B accuracy 0.7833, KL 0.0952")
for arm in results:
    td = results[arm]["td_test"]
    print(f"  {arm}: accuracy {td['top_choice']:.4f}, KL {td['kl']:.4f} (uniform {td['uniform_kl']:.4f})")

os.makedirs(CONFIG["out_dir"], exist_ok=True)
table.to_csv(os.path.join(CONFIG["out_dir"], "results.csv"), index=False)
pd.DataFrame(history).to_csv(os.path.join(CONFIG["out_dir"], "history.csv"), index=False)
qs_report.to_csv(os.path.join(CONFIG["out_dir"], "qs_check.csv"), index=False)
print("saved", CONFIG["out_dir"])